<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/rickiepark/nlp-with-transformers/blob/main/08_model-compression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [ ]:
# 코랩이나 캐글을 사용한다면 이 셀의 주석을 제거하고 실행하세요.
!git clone https://github.com/kikim6114/nlp-practice-2025.git
%cd nlp-practice-2025
from install import *
install_requirements(chapter=8)

In [ ]:
!pip uninstall gcsfs
!pip install gcsfs==2025.3.0
!pip install fsspec==2025.3.0
!pip install -U huggingface_hub datasets

# 8. 효율적인 트랜스포머 구축

#### 작지만 빠르고 정확도 높은 모델이 필요하다면?
#### 예측 속도를 높이고 트랜스포머 모델의 메모리 사용량을 줄이는 기술 4 가지:
- 지식 정제(Knowledge Distillation)
- 양자화(Quantization)
- 가지치기(Pruning)
- ONNX 포맷과 ONNX 런타임(ORT)을 사용한 그래프 최적화(Graph Optimization)

<img alt="Scaling BERT at Roblox" caption="How Roblox scaled BERT with knowledge distillation, dynamic padding, and weight quantization (photo courtesy of Roblox employees Quoc N. Le and Kip Kaehler)" src="https://github.com/kikim6114/nlp-practice-2025/blob/main/images/chapter08_roblox.png?raw=1" width="800" id="roblox"/>
<p align="center">[그림 8-1] Roblox가 지식 정제, 동적 패딩, 가중치 양자화로 BERT를 스케일링한 예(로블록스 직원인 Quoc N. Le와 Kip Kaehler 제공)
</p>
<br>
* 지식 정제와 양자화를 연결해 레이턴시와 BERT 분류기의 처리 성능을 30배 이상 향상시킴

## 8.1 의도 탐지 예제
- 금융사 콜센터 전용 텍스트 기반 어시스턴스 시스템 구축하기
- 의도 탐지(Intent Detection):
  - 텍스트 기반 어시스턴스 시스템에서 중요한 구성 요소
  - 사용자 쿼리를 사전에 정의된 일련의 행동이나 의도로 분류하고 후속 작업(예: 렌트가 예약이면 렌트가 예약 관련 대화로 전환)
  - 대화 유지를 위해서 latency가 낮아야 한다
  - 사용자가 정의된 의도에 속하지 않은 쿼리를 제공하면 이에 대하여 응답을 출력해야 한다.

<img alt="Out of Scope Query" width="300" caption="Three exchanges between a human (right) and a text-based assistant (left) for personal finance (courtesy of Stefan Larson et al.)" src="https://github.com/rickiepark/nlp-with-transformers/blob/main/images/chapter08_oos.png?raw=1" id="oos"/>
<p>[그림 8-2] 사람(오른쪽)과 텍스트 기반 어시스턴트(왼쪽)가 개인 금융정보에 대해 나눈 대화(Stefan Larson 등이 제공)
</p>

##### 기준 모델: CLINIC150 dataset에서 fine-tuning된 BERT base 모델(정확도: 94%)
##### CLINIC150 dataset:
- 150 개 intent classes
- 22,500 queries for 10 개 domains
- 1,200 queries for oos class(Out-Of-Scope class)

In [ ]:
from transformers import pipeline
# 허깅페이스 허브에서 미세 튜닝한 모델을 다운로드 하고 위 사진의 태스크를 진행
bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)

In [ ]:
query = """Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th in
Paris and I need a 15 passenger van"""
pipe(query)

## 8.2 벤치마크 클래스 만들기
#### 트랜스포머를 제품 환경에 배포하기 위해 절충해야 할 제약 조건들:
- 모델 성능
- 레이턴시
- 메모리  


##### 이러한 제약들을 다양한 압축 기법으로 최적화하는 방법을 알아본다.

In [ ]:
class PerformanceBenchmark:
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        self.dataset = dataset
        self.optim_type = optim_type

    def compute_accuracy(self):
        # 나중에 정의합니다
        pass

    def compute_size(self):
        # 나중에 정의합니다
        pass

    def time_pipeline(self):
        # 나중에 정의합니다
        pass

    def run_benchmark(self):
        metrics = {}
        metrics[self.optim_type] = self.compute_size()
        metrics[self.optim_type].update(self.time_pipeline())
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics

#### 테스트셋에서 모델 정확도 계산

In [ ]:
from datasets import load_dataset

clinc = load_dataset("clinc_oos", "plus")  # "plus": oos 훈련샘플이 담긴 subset

In [ ]:
clinc

In [ ]:
import pandas as pd
pd.DataFrame(clinc["test"][40:45])

In [ ]:
sample = clinc["test"][42]
sample

In [ ]:
clinc["test"].features["intent"]

In [ ]:
len(clinc["test"].features["intent"].names)

`ClassLabel`: HuggingFace의 datasets 라이브러리에서 사용되는 기능. 범주형 데이터가 포함된 데이터셋의 열을 나타내는 데 사용되며, 각 범주는 레이블과 연관됨. 레이블은 내부적으로 정수로 저장되며, 정수 표현과 문자열 레이블 간 변환을 위한 메서드 제공.

In [ ]:
intents = clinc["test"].features["intent"]
intents.int2str(sample["intent"])

#### `PerformanceBenchmark`의 `computer_accuracy()` 구현
##### 🤗 [`evaluate`](https://pypi.org/project/evaluate/):  모델을 평가하고 비교하며, 성능 리포트를 보다 쉽고 표준화된 방식으로 만들어 주는 라이브러리

In [ ]:
!pip install evaluate

In [ ]:
import evaluate

accuracy_score = evaluate.load("accuracy")  # 정확도 metric 로드

In [ ]:
def compute_accuracy(self):
    """PerformanceBenchmark.compute_accuracy() 메서드를 오버라이드"""
    preds, labels = [], []
    for example in self.dataset:
        pred = self.pipeline(example["text"])[0]["label"]  # pipeline 출력 내용 예:  [{'label': 'car_rental', 'score': 0.549003541469574}]
        label = example["intent"]
        preds.append(intents.str2int(pred))
        labels.append(label)
    accuracy = accuracy_score.compute(predictions=preds, references=labels)
    print(f"테스트 세트 정확도 - {accuracy['accuracy']:.3f}")
    return accuracy

PerformanceBenchmark.compute_accuracy = compute_accuracy

##### `torch.save()` 함수를 사용해 모델을 디스크에 직렬화하고 크기를 계산  
##### `state_dict()` 메소드는 모델의 층과 학습가능 파라미터를 매핑하는 딕셔너리 반환  

In [ ]:
list(pipe.model.state_dict().items())[42]

In [ ]:
for i, param_tensor in enumerate(pipe.model.state_dict()):
    print(f"{i}\t{param_tensor}\t{pipe.model.state_dict()[param_tensor].size()}")

#### 모델 저장 방법:
```python
torch.save(pipe.model.state_dict(), "model.pt")
```

In [ ]:
import torch
from pathlib import Path

def compute_size(self):  # 모델 크기를 계산하는 메소드
    """PerformanceBenchmark.compute_size() 메서드를 오버라이드"""
    state_dict = self.pipeline.model.state_dict()
    tmp_path = Path("model.pt")
    torch.save(state_dict, tmp_path)
    # 메가바이트 단위로 크기를 계산
    # Path(tmp_path).stat().st_size : 바이트 단위 크기
    size_mb = Path(tmp_path).stat().st_size / (1024 * 1024)
    # 임시 파일을 삭제
    tmp_path.unlink()
    print(f"모델 크기 (MB) - {size_mb:.2f}")
    return {"size_mb": size_mb}

PerformanceBenchmark.compute_size = compute_size

#### 평균 레이턴시를 측정하기 위한 `time_pipeline()` 함수 구현:
- `time.perf_counter()` : `time.time()` 보다 정밀

In [ ]:
from time import perf_counter

for _ in range(3):
    start_time = perf_counter()
    _ = pipe(query)
    latency = perf_counter() - start_time
    print(f"레이턴시 (ms) - {1000 * latency:.3f}")

- 레이턴시 편차가 심하므로, 평균과 표준편차로 분석할 필요가 있다.

In [ ]:
import numpy as np

def time_pipeline(self, query="What is the pin number for my account?"):
    """PerformanceBenchmark.time_pipeline() 메서드를 오버라이드"""
    latencies = []
    # 워밍업
    for _ in range(10):
        _ = self.pipeline(query)
    # 실행 측정
    for _ in range(100):
        start_time = perf_counter()
        _ = self.pipeline(query)
        latency = perf_counter() - start_time
        latencies.append(latency)
    # 통계 계산
    time_avg_ms = 1000 * np.mean(latencies)
    time_std_ms = 1000 * np.std(latencies)
    print(f"평균 레이턴시 (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
    return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

PerformanceBenchmark.time_pipeline = time_pipeline

In [ ]:
pb = PerformanceBenchmark(pipe, clinc["test"])
perf_metrics = pb.run_benchmark()

## 8.3 지식 정제로 모델 크기 줄이기
- 느리고 크지만 성능이 더 높은 티처(teacher)의 동작을 모방하도록 작은 스튜던트(student) 모델을 훈련
- 2006년 앙상블 모델을 위해 처음 소개됨
- 2015년 신경망에 적용(Hinton et al., "Distilling the Knowledge in a Neural Network") - 어렵지 않음. 읽어 볼 것.

### 8.3.1 미세 튜닝에서의 지식 정제

<img alt="Soft Probabilities" caption="Comparison of a hard label that is one-hot encoded (left), softmax probabilities (middle), and softened class probabilities (right)" src="https://github.com/rickiepark/nlp-with-transformers/blob/main/images/chapter08_soft-probs.png?raw=1" width="800" id="soft-probs"/>

<img alt="Knowledge distillation" caption="The knowledge distillation process" src="https://github.com/rickiepark/nlp-with-transformers/blob/main/images/chapter08_kd.png?raw=1" width="800" id="kd"/>

- 티처의 소프트 확률로 정답 레이블을 보강하여 스튜던트가 학습할 때 부가 정보를 제공한다. 예를 들어 여러 개의 의도(intent)에 높은 확률을 할당하다면 이 의도들은 특성 공간 안에서 서로 가까이 위치 한다는 신호일 가능성이 존재하고 이런 확률을 모방하도록 훈련하여 레이블만으로 얻지 못하는 지식(결정 경계에 대한 정보)까지 학습하도록 한다.
- Dark Knowledge: 레이블 만으로는 얻지 못하는 지식(by Geoffrey Hinton)

### 8.3.2 사전 훈련에서의 지식 정제

### 8.3.3 지식 정제 트레이너 만들기
#### `TrainingArguments`
- `transformers.Trainer` 클래스와 함께 사용되며, 모델 학습의 다양한 설정을 정의한다.
- 학습 과정에서 필요한 파라미터(예: 학습률, 배치 크기, 에포크 수 등)를 지정하고, 학습 로그, 모델 저장, 평가 전략 등을 커스터마이징할 수 있다.

#### 지식 정제를 구현하기 위해 `Trainer` 클래스에 속성 추가
- `TrainingArguments`를 상속해서 몇 가지 속성 추가

In [ ]:
from transformers import TrainingArguments

class DistillationTrainingArguments(TrainingArguments):
    def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.temperature = temperature

In [ ]:
from dataclasses import dataclass, field
from transformers import TrainingArguments

@dataclass
class DistillationTrainingArguments(TrainingArguments):
    alpha: float = field(
        default=0.5,
        metadata={"help": "CrossEntropy와 Distillation Loss의 가중치 비율 (0.0~1.0)"}
    )
    temperature: float = field(
        default=2.0,
        metadata={"help": "logits을 부드럽게(soften) 할 온도 (T > 1)"}
    )

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class DistillationTrainer(Trainer): # 지식 정제 손실항 추가
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model  # 미리 학습된 모델 (티처 모델)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        inputs = inputs.to(device)
        # Student 모델 순전파 → CrossEntropy 손실, Student 로짓 얻기
        outputs_stu = model(**inputs)
        # 스튜던트의 크로스 엔트로피 손실과 로짓을 추출합니다
        loss_ce = outputs_stu.loss
        logits_stu = outputs_stu.logits
        # 티처의 로짓을 추출합니다
        with torch.no_grad():
            outputs_tea = self.teacher_model(**inputs)
            logits_tea = outputs_tea.logits
        # 확률을 부드럽게하고 teacher 모델의 정보를 반영
        # alpha를 조절해서 라벨에 대한 정보와 teacher 정보의 비율을 조절
        loss_fct = nn.KLDivLoss(reduction="batchmean") # KL-divergence를 계산해 주는 손실 함수
        loss_kd = self.args.temperature ** 2 * loss_fct( # final loss에서 균형잡힌 계산을 맞추기 편해짐
            F.log_softmax(logits_stu / self.args.temperature, dim=-1),
            F.softmax(logits_tea / self.args.temperature, dim=-1))
        # 가중 평균된 스튜던트 손실을 반환합니다
        loss = self.args.alpha * loss_ce + (1. - self.args.alpha) * loss_kd
        return (loss, outputs_stu) if return_outputs else loss

### 8.3.4 좋은 스튜던트 초기화 선택하기

In [ ]:
from transformers import AutoTokenizer

student_ckpt = "distilbert-base-uncased"
student_tokenizer = AutoTokenizer.from_pretrained(student_ckpt)

def tokenize_text(batch):
    return student_tokenizer(batch["text"], truncation=True)

clinc_enc = clinc.map(tokenize_text, batched=True, remove_columns=["text"])
clinc_enc = clinc_enc.rename_column("intent", "labels")  # 미세튜닝할 때 Trainer는 기본적으로 labels 이름의 열을 찾는다.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
def compute_metrics(pred): # 추적할 성능 지표
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_score.compute(predictions=predictions, references=labels)

In [ ]:
# batch_size = 48

# finetuned_ckpt = "distilbert-base-uncased-finetuned-clinc"
# student_training_args = DistillationTrainingArguments(
#     output_dir=finetuned_ckpt, eval_strategy = "epoch",  # correct "evaluation_strategy" to "eval_strategy"
#     num_train_epochs=5, learning_rate=2e-5,
#     per_device_train_batch_size=batch_size,
#     per_device_eval_batch_size=batch_size, alpha=1, weight_decay=0.01,
#     push_to_hub=True)

In [ ]:
# This cell was used when above cell raised an error
batch_size = 48
finetuned_ckpt = "distilbert-base-uncased-finetuned-clinc"
# alpha를 1로 해서 모델 튜닝 시작
student_training_args = DistillationTrainingArguments(
    # output_dir="dummy_dir",
    output_dir=finetuned_ckpt,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",   # ← WandB, TensorBoard 등 모든 로깅 끔
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    alpha=1.0,
    weight_decay=0.01,
    # push_to_hub=True,
    push_to_hub=False,
)

In [ ]:
student_training_args.logging_steps = len(clinc_enc['train']) // batch_size
student_training_args.disable_tqdm = False
student_training_args.save_steps = 1e9
# 트랜스포머 4.23.0 버전부터 로깅 수준을 문자열로 지정해야 합니다.
# student_training_args.log_level = 40
student_training_args.log_level = 'error'

In [ ]:
%env TOKENIZERS_PARALLELISM=false

In [ ]:
id2label = pipe.model.config.id2label # 의도
label2id = pipe.model.config.label2id # 라벨

In [ ]:
from transformers import AutoConfig

num_labels = intents.num_classes  # 모델이 기대해야할 클래스 갯수 설정
student_config = (AutoConfig
                  .from_pretrained(student_ckpt, num_labels=num_labels,
                                   id2label=id2label, label2id=label2id))

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 위에서 설정한 것들을 아래 AutoModelForSequenceClassification 클래스의 from_pretrained 함수에 전달
def student_init(): # 트레이너로 여러 번 실행할 거라서 모델 초기화 함수 필요
    return (AutoModelForSequenceClassification
            .from_pretrained(student_ckpt, config=student_config).to(device))

In [ ]:
teacher_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
teacher_model = (AutoModelForSequenceClassification
                 .from_pretrained(teacher_ckpt, num_labels=num_labels)
                 .to(device))

In [ ]:
distilbert_trainer = DistillationTrainer(model_init=student_init,
    teacher_model=teacher_model, args=student_training_args,
    train_dataset=clinc_enc['train'], eval_dataset=clinc_enc['validation'],
    compute_metrics=compute_metrics, tokenizer=student_tokenizer)

distilbert_trainer.train()

In [ ]:
distilbert_trainer.push_to_hub("Training completed!")

In [ ]:
# `kikim6114`를 자신의 허브 사용자 이름으로 바꾸세요.
finetuned_ckpt = "kikim6114/distilbert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=finetuned_ckpt)

In [ ]:
optim_type = "DistilBERT"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
def plot_metrics(perf_metrics, current_optim_type):
    df = pd.DataFrame.from_dict(perf_metrics, orient='index')

    for idx in df.index:
        df_opt = df.loc[idx]
        # 현재 최적화 방법을 점선으로 그립니다
        if idx == current_optim_type:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        alpha=0.5, s=df_opt["size_mb"], label=idx,
                        marker='$\u25CC$')
        else:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        s=df_opt["size_mb"], label=idx, alpha=0.5)

    legend = plt.legend(bbox_to_anchor=(1,1))
    for handle in legend.legend_handles:
        handle.set_sizes([20])

    plt.ylim(80,90)
    # 가장 느린 모델을 사용해 x 축 범위를 정합니다
    xlim = int(perf_metrics["BERT baseline"]["time_avg_ms"] + 3)
    plt.xlim(1, xlim)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Average latency (ms)")
    plt.show()

plot_metrics(perf_metrics, optim_type)

### 8.3.5 옵투나로 좋은 하이퍼파라미터 찾기
- 좋은 $\alpha$ 와 $\mathit{T}$ 값을 찾기 위해 2D 하이퍼파라미터 공간에서 그리드 서치(grid search)를 수행할 수 있다.
- 최적화 프레임워크 Optuna를 사용하는 것이 더 좋은 방법이다.
- Optuna는 search 문제를 여러 시도를 통해 최적화시킬 목적 함수로 표현한다.

##### 로젠브록의 바나나 함수 최소화:
$$f(x, y)=(1-x)^2+100(y-x^2)^2$$

In [ ]:
# 두 변수를 가진 로젠브록(Rosenbrock) 함수 그래프
import matplotlib.pyplot as plt
import numpy as np

def f(x, y):
    return (1-x)**2+100*(y-x**2)**2

X, Y = np.meshgrid(np.linspace(-2, 2, 250), np.linspace(-1, 3, 250))
Z = f(X,Y)
_, ax = plt.subplots()
ax.plot([1], [1], 'x', mew=3, markersize=10, color="red")
ax.contourf(X, Y, Z, np.logspace(-1, 3, 30), cmap='viridis', extend="both")
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-0.9, 1.7)
plt.show()

이 바나나 함수는 점 $(1,1)$에서 최솟값을 가지며, 골짜기 영역을 찾는 것은 쉽지만 최솟값으로 수렴하기는 어렵다는 특징을 갖는다.

In [ ]:
def objective(trial):
    x = trial.suggest_float("x", -2, 2)
    y = trial.suggest_float("y", -2, 2)
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

- `suggest_int`와 `suggest_catagorical` 도 제공

In [ ]:
import optuna

study = optuna.create_study()
study.optimize(objective, n_trials=1000)

In [ ]:
study.best_params

#### 트랜스포머스에서 옵투나 사용 순서
1) 최적화 하려는 하이퍼파라미터 공간을 정의   
2) Trainer의 hyperparameter_search() 메서드에 시도 횟수, 최적화 방향, 하이퍼라라미터 검색 공간을 전달  
3) BestRun 반환됨  


In [ ]:
def hp_space(trial):
    return {"num_train_epochs": trial.suggest_int("num_train_epochs", 5, 10),
        "alpha": trial.suggest_float("alpha", 0, 1),
        "temperature": trial.suggest_int("temperature", 2, 20)}

In [ ]:
best_run = distilbert_trainer.hyperparameter_search(
    n_trials=20, direction="maximize", hp_space=hp_space)

In [ ]:
print(best_run)

In [ ]:
for k,v in best_run.hyperparameters.items():
    setattr(student_training_args, k, v)

# 정제된 모델을 저장할 새로운 저장소를 정의합니다
distilled_ckpt = "distilbert-base-uncased-distilled-clinc"
student_training_args.output_dir = distilled_ckpt

# 최적의 매개변수로 새로운 Trainer를 만듭니다
distil_trainer = DistillationTrainer(model_init=student_init,
    teacher_model=teacher_model, args=student_training_args,
    train_dataset=clinc_enc['train'], eval_dataset=clinc_enc['validation'],
    compute_metrics=compute_metrics, tokenizer=student_tokenizer)

distil_trainer.train();

In [ ]:
distil_trainer.push_to_hub("Training complete")

### 정제 모델 벤치마크 수행하기

In [ ]:
# `kikim6114`를 자신의 허브 사용자 이름으로 바꾸세요.
distilled_ckpt = "kikim6114/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)
optim_type = "Distillation"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

## 8.4 양자화로 모델 속도 높이기

### 사이드바: 부동 소수점 숫자 vs 고정 소수점 숫자
- 고정소수점: $123.456$
- 부동소수점: $1.23456 \times 10^2=0.00123456 \times 10^5=12.3456 \times 10$ ... 등
- 가중치, 활성화, 그래디언트 등은 FP32 또는 FP16과 FP32를 혼합해 표현한다.
#### 고정 소수점



### End sidebar

<img alt="Mapping floating-point numbers to 8-bit integers" width="800" caption="Quantizing floating-point numbers as unsigned 8-bit integers (courtesy of Manas Sahni)" src="https://github.com/kikim6114/nlp-practice-2025/blob/main/images/chapter08_fp32-to-int8.png?raw=1" id="fp32toint8"/> <br>
[그림 8-6] 부동소수점 숫자를 부호 없는 8비트 정수로 양자화하기(Manas Sahni 제공)

#### 양자화의 기본 원리:   
부동소수점 숫자 $f$를 원래 범위 $[f_{max}, f_{min}]$에서 고정소수점 숫자 $q$로 표현된 작은 범위 $[q_{max}, q_{min}]$으로 매핑하고 모든 값을 이 구간에 선형적으로 분포시킨다.
$$f=\left(\frac{f_{max}-f_{min}}{q_{max}-q_{min}}\right)(q-Z)=S(q-Z)$$
여기서,
  - $S$: scale factor
  - $Z$: $q$와 동일한 타입으로, $f=0$ 을 양자화한 값으로서 영점(zero point)라고 함.

In [ ]:
import matplotlib.pyplot as plt

state_dict = pipe.model.state_dict()
weights = state_dict["distilbert.transformer.layer.0.attention.out_lin.weight"]
plt.hist(weights.flatten().cpu().numpy(), bins=250, range=(-0.3,0.3), edgecolor="C0")
plt.show()

8-bit 정수로 압축한다면,
$[q_{max}, q_{min}]=[-128, 127]$

In [ ]:
zero_point = 0
scale = (weights.max() - weights.min()) / (127 - (-128))

양자화된 텐서 계산:
$$q=\frac{f}{S}+Z$$
를 계산하고, 범위 밖의 값을 클리핑하고, 가장 가까운 정수로 반올림하고, `Tensor.char()` 함수를 사용해 `torch.int8` 타입으로 표현.

In [ ]:
(weights / scale + zero_point).clamp(-128, 127).round().char()

#### 파이토치의 함수 사용한 양자화:

In [ ]:
from torch import quantize_per_tensor

dtype = torch.qint8
quantized_weights = quantize_per_tensor(weights, scale, zero_point, dtype)
quantized_weights.int_repr()

In [ ]:
# 트랜스포머 가중치에서 양자화 효과
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes,mark_inset

# 히스토그램 그리기
fig, ax = plt.subplots()
ax.hist(quantized_weights.dequantize().flatten().cpu().numpy(),
         bins=250, range=(-0.3,0.3), edgecolor="C0");
# 확대 그림 만들기
axins = zoomed_inset_axes(ax, 5, loc='upper right')
axins.hist(quantized_weights.dequantize().flatten().cpu().numpy(),
         bins=250, range=(-0.3,0.3));
x1, x2, y1, y2 = 0.05, 0.1, 500, 2500
axins.set_xlim(x1, x2)
axins.set_ylim(y1, y2)
axins.axes.xaxis.set_visible(False)
axins.axes.yaxis.set_visible(False)
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")
plt.show()

In [ ]:
%%timeit
weights @ weights  # FP32 @ FP32

In [ ]:
weights_cpu = weights.cpu()

In [ ]:
%%timeit
weights_cpu @ weights_cpu  # FP32 @ FP32

##### 양자화된 텐서 곱

In [ ]:
from torch.nn.quantized import QFunctional

q_fn = QFunctional()

# Move the quantized tensor to CPU before using QFunctional
quantized_weights_cpu = quantized_weights.cpu()

In [ ]:
%%timeit
q_fn.mul(quantized_weights_cpu, quantized_weights_cpu)

In [ ]:
1.45*1000/55.5

가중치 텐서와 양자화된 텐서의 저장 크기 비교

In [ ]:
import sys

sys.getsizeof(weights.untyped_storage()) / sys.getsizeof(quantized_weights.untyped_storage())

트랜스포머의 실제 압축 비율은 어떤 층을 양자화하는지에 따라 달라지며, 보통 선형 층만 양자화 한다. 모든 연산에서 정밀도를 바꾸면 모델 성능에 나쁜 영향을 주므로 주의해야 한다.
#### 신경망에서의 양자화 방법
1) 동적 양자화:
  - 훈련 중에는 적용하지 않으며, 가중치와 활성화값이 추론 직전에 (동적으로) INT8으로 변환
  - 가장 간단한 방법
  - 활성화를 메모리에 읽고 쓸 때 부동소수점 형식을 사용하므로 정밀도 변환이 병목이 될 수 있다.

2) 정적 양자화:
  - 양자화 체계를 사전에 계산해 양자화 변환 부담을 피하는 방법
  - 추론에 앞서 대표적 샘플 데이터에서 활성화 패턴을 관찰하여 수행
  - 대표적 샘플을 찾는 방법이 어려움
  - 훈련과 추론 과정애서 정밀도 차이로 인한 모델 성능 저하 문제 해결이 어려움   

3) 양자화를 고려한 훈련:   
  - 가짜로 FP32 값을 양자화해 훈련 중에 양자화 효과를 시뮬레이션
  - 훈련 중 FP32를 반올림하여 양자화 효과를 흉내낸다.
  - 정방향/역방향 패스 모두에 적용됨 정적/동적 양자화를 모두 사용해 모델 성능을 높인다.

#### 양자화 방법의 선택 기준
- 추론 시 트랜스포머의 주요 병목 지점은 가중치 관련 연산 및 메모리 대역폭
- 따라서 트랜스포머 기반 NLP 모델에서는 동적 양자화가 최상의 선택
- 보다 작은 컴퓨터 비전 모델에서는 정적 양자화
- 병목 현상이 너무 크면 양자화를 고려한 훈련을 사용

#### PyTorch에서 동적 양자화

In [ ]:
from torch.quantization import quantize_dynamic

# `kikim6114`를 자신의 허브 사용자 이름으로 바꾸세요.
model_ckpt = "kikim6114/distilbert-base-uncased-distilled-clinc"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = (AutoModelForSequenceClassification
         .from_pretrained(model_ckpt).to("cpu"))

model_quantized = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

### 8.5 양자화된 모델의 벤치마크 수행하기

In [ ]:
pipe = pipeline("text-classification", model=model_quantized,
                tokenizer=tokenizer, device="cpu")
optim_type = "Distillation + quantization"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

## 8.6 ONNX와 ONNX 런타임으로 추론 최적화하기

#### ONNX(Open Neural Network Exchange) 란?

- 파이토치, 텐서플로 등 다양한 프레임워크에서 딥러닝 모델을 나타내기 위해 공통 연산자와 공통 파일 포맷을 정의하는 공개 표준.
- ONNX는 모델을 중간 계층 형식으로 변환하고 모델을 실행하는 데 필요한 라이브러리를 제공한다.
- 파이토치 모델 $\leftrightarrow$ ONNX $\leftrightarrow$ 텐서플로 모델 임포트 가능

#### ONNX Runtime 이란?

- ONNX 모델을 실행하기 위한 엔진으로서, ONNX 런타임은 빠른 추론을 위한 최적화된 커널을 사용한다.
- CPU, GPU 및 딥러닝 가속기(DNNL, NNAPI, OpenVINO)를 지원.

#### ONNX가 만들어 내는 계산 그래프
- 모델을 ONNX 포맷으로 내보내면 ONNX의 연산자를 사용해 신경망을 통과하는 데이터의 흐름을 표현하기 위해 계산그래프(중간 표현이라고도 함)를 만든다.

<img alt="Example ONNX graph" width="500" caption="A section of the ONNX graph for BERT-base, visualized in Netron" src="https://github.com/kikim6114/nlp-practice-2025/blob/main/images/chapter08_bert-onnx.png?raw=1" id="bert-onnx"/>
<br>
[그림 8-8] Netron으로 시각화한 BERT 기반 모델에 대한 ONNX 그래프의 한 섹션


#### ONNX 런타임
- ONNX 런타임(ORT)는 연산자 융합 상수 폴딩 같은 ONNX 그래프 최적화 도구를 제공
- 여러 종류의 하드웨어에서 모델을 실행하도록 실행 공급자(execution provide)를 위한 인터페이스를 정의하며 이것은 강력한 추상화이다.

<img alt="Architecture of the ONNX and ONNX Runtime ecosystem" width="500" caption="Architecture of the ONNX and ONNX Runtime ecosystem (courtesy of the ONNX Runtime team)" src="https://github.com/kikim6114/nlp-practice-2025/blob/main/images/chapter08_onnx-ort.png?raw=1" id="onnx-ort"/> <br>
[그림 8-9] ONNX와 ONNX 런타임 생태계의 아키텍처(ONNX 런타임 팀 제공)

#### ONNX 실행을 위한 `convert_graph_to_onnx.convert()` 함수가 처리하는 일:
1) 하나의 파이프라인으로 모델을 초기화  
2) ONNX가 계산 그래프를 기록하도록 플레이스홀더 입력으로 파이프라인을 실행  
3) 동적 시퀀스 길이를 처리하기 위해 동적인 축을 정의  
4) 네트워크 파라미터와 함께 이 그래프를 저장  

#### ONNX를 위한 OpenMP 환경 변수를 지정:

In [ ]:
!pip install onnx
!pip install onnxruntime

In [ ]:
import os
from psutil import cpu_count
# from transformers import AutoModelForSequenceClassification, AutoTokenizer # 임시로 작성
# from pathlib import Path

os.environ["OMP_NUM_THREADS"] = f"{cpu_count()}"
os.environ["OMP_WAIT_POLICY"] = "ACTIVE"

In [ ]:
from transformers.convert_graph_to_onnx import convert

# "kikim6114"를 자신의 허브 사용자 이름으로 바꾸세요.
model_ckpt = "kikim6114/distilbert-base-uncased-distilled-clinc"
model = (AutoModelForSequenceClassification
         .from_pretrained(model_ckpt).to("cpu")) # Move model to CPU
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
onnx_model_path = Path("onnx/model.onnx")
convert(framework="pt", model=model_ckpt, tokenizer=tokenizer,
        output=onnx_model_path, opset=12, pipeline_name="text-classification")

모델을 저장했으므로, 모델에 입력을 전달하기 위해 `InferenceSession` 객체를 만든다.

In [ ]:
from onnxruntime import (GraphOptimizationLevel, InferenceSession,
                         SessionOptions)

def create_model_for_provider(model_path, provider="CPUExecutionProvider"):
    options = SessionOptions()
    options.intra_op_num_threads = 1
    options.graph_optimization_level = GraphOptimizationLevel.ORT_ENABLE_ALL
    session = InferenceSession(str(model_path), options, providers=[provider])
    session.disable_fallback()
    return session

In [ ]:
onnx_model = create_model_for_provider(onnx_model_path)

In [ ]:
inputs = clinc_enc["test"][:1]
del inputs["labels"]  # ONNX는 input_ids와 attention_mask 만 입력으로 기대하므로 label 열은 삭제
logits_onnx = onnx_model.run(None, inputs)[0]
logits_onnx.shape

In [ ]:
np.argmax(logits_onnx)

In [ ]:
clinc_enc["test"][0]["labels"]

ONNX 모델은 text_classification 파이프라인과 호환되지 않으므로 핵심 동작을 흉내 재는 사용자 정의 클래스를 만들자.

In [ ]:
from scipy.special import softmax

class OnnxPipeline:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def __call__(self, query):
        model_inputs = self.tokenizer(query, return_tensors="pt")
        inputs_onnx = {k: v.cpu().detach().numpy()
                       for k, v in model_inputs.items()}
        logits = self.model.run(None, inputs_onnx)[0][0, :]
        probs = softmax(logits)
        pred_idx = np.argmax(probs).item()
        return [{"label": intents.int2str(pred_idx), "score": probs[pred_idx]}]

In [ ]:
pipe = OnnxPipeline(onnx_model, tokenizer)
pipe(query)

In [ ]:
class OnnxPerformanceBenchmark(PerformanceBenchmark):
    def __init__(self, *args, model_path, **kwargs):
        super().__init__(*args, **kwargs)
        self.model_path = model_path

    def compute_size(self):
        size_mb = Path(self.model_path).stat().st_size / (1024 * 1024)
        print(f"모델 크기 (MB) - {size_mb:.2f}")
        return {"size_mb": size_mb}

In [ ]:
optim_type = "Distillation + ORT"
pb = OnnxPerformanceBenchmark(pipe, clinc["test"], optim_type,
                              model_path="onnx/model.onnx")
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

model_input = "onnx/model.onnx"
model_output = "onnx/model.quant.onnx"
quantize_dynamic(model_input, model_output, weight_type=QuantType.QInt8)

In [ ]:
onnx_quantized_model = create_model_for_provider(model_output)
pipe = OnnxPipeline(onnx_quantized_model, tokenizer)
optim_type = "Distillation + ORT (quantized)"
pb = OnnxPerformanceBenchmark(pipe, clinc["test"], optim_type,
                              model_path=model_output)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

## 8.7 가중치 가지치기로 희소한 모델 만들기
신경망에서 가장 덜 중요한 가중치를 찾아 제거하는 방식으로 모델의 파라미터 개수를 줄이는 방법

### 8.7.1 심층 신경망의 희소성
- 모델이 점차 희소화되며, 희소 행렬 포맷으로 저장된다
- 지치기와 양자화를 함께 사용하면 추가 압축이 가능하다

<img alt="Network Pruning" width="500" caption="Weights and neurons before and after pruning (courtesy of Song Han)" src="https://github.com/kikim6114/nlp-practice-2025/blob/main/images/chapter08_network-pruning.png?raw=1" id="network-pruning"/> <br>
[그림 8-10] 가지치기 전후의 가중치와 뉴런(Song Han 제공)

### 8.7.1 가중치 가지치기 방법


$$\text{Top}_k(\mathbf{S})_{ij}=\begin{cases} 1 & S_{ij}가 \, 최상위\, k \%인\, 경우, \\ 0 & 그\, 외의\, 경우 \end{cases} $$

- $\mathbf{S}$: 중요도 점수(importance scores) 행렬
- $k$: 희소성 비율을 제어하는 하이퍼파라미터($k$가 작을수록 행렬이 더 희소해진다).
- 이 점수를 사용해 마스크 행렬(mask matrix) $\mathbf{M}$을 만든다. 이 행렬은 정방향 패스에서 입력 $x_i$에 곱해지는 가중치 $W_{ij}$를 마스킹해 희소한 활성화 $a_i$를 만든다:

$$a_i = \sum_{j}{W_{ij}M_{ij}x_j}$$


#### 가지치기에서 고려할 것들
- 어떤 가중치를 삭제할 것인가?
- 최상의 성능을 내려면 남은 가중치를 어떻게 조정해야 하는가?
- 이런 가지치기 계산을 효율적으로 수행하는 방법은 무엇인가?

#### 절댓값 가지치기
1. 중요도 점수 행렬을 학습한다: $ \mathbf{S}=(|W_{ij}|)_{1 \le j,\, j \le n} $
2. 마스크 행렬을 만든다: $\mathbf{\text M}=\text{Top}_k(\mathbf{S})$  
3. 원하는 희소성에 도달하지 않았으면 1 단계로 간다.

#### 가지치기에 사용되는 세제곱 희소성 스케줄러
- 희소성 초기값 $s_i$에서 출발하여 매 $\Delta t$ 스텝마다 $s_i$를 증가시켜 $N$ 단계 후에 희소성 최종값 $s_f$ 도달하게 한다.  
- 매 $\Delta t$ 스텝마다 이진 마스크 $\mathbf{\text M}$ 을 업데이트해 훈련하는 동안 마스킹된 가중치를 다시 활성화하고 가지치기로 인해 발행할지 모를 잠재적 정확도 손실을 복구하는 방법을 사용한다:

$$ s_t = s_f + (s_i - s_f)\left(1-\frac{t-t_0}{N\Delta t}\right)^3, \,\,\,\, t \in \{t_0, t_0+\Delta t, ..., N\Delta t \}$$

In [ ]:
# 가지치기에 사용되는 세제곱 희소성 스케줄러
import numpy as np
import matplotlib.pyplot as plt

def _sparsity(t, t_0=0, dt=1, s_i=0, s_f=0.9, N=100):
    return s_f + (s_i - s_f) * (1 - (t - t_0) / (N * dt))**3

steps = np.linspace(0,100,100)
values = [_sparsity(t) for t in steps]

fig, ax = plt.subplots()
ax.plot(steps, values)
ax.set_ylim(0,1)
ax.set_xlim(0,100)
ax.set_xlabel("Pruning step")
ax.set_ylabel("Sparsity")
plt.grid(linestyle="dashed")
plt.show()

전이 학습에서는 가중치의 중요도가 주로 사전 훈련 단계에서 결정된다. 따라서 절댓값 가지치기로 인해 미세 튜닝 작업에서 중요한 가중치가 삭제될 수도 있다.

#### 이동 가지치기
- 미세튜닝하는 동안 점진적으로 가중치를 제거해 모델을 점차 희소하게 만든다.
- 미세튜닝 동안 가중치와 점수가 모두 학습된다
- 원점으로부터 가장 크게 이동하는 가중치가 가장 중요한 가중치다.


<img alt="Magnitude vs Movement Pruning" width="700" caption="Comparison of weights removed (in gray) during magnitude pruning (left) and movement pruning (right)" src="https://github.com/kikim6114/nlp-practice-2025/blob/main/images/chapter08_magnitude-vs-movement.png?raw=1" id="magnitude-vs-movement"/> <br>
[그림 8-12] 절댓값 가지치기(왼쪽)와 이동 가지치기(오른쪽)에서 삭제된 가중치 비교

<img alt="Pruning Distributions" width="500" caption="Distribution of remaining weights for magnitude pruning (MaP) and movement pruning (MvP)" src="https://github.com/kikim6114/nlp-practice-2025/blob/main/images/chapter08_pruning-dists.png?raw=1" id="pruning-dists"/> <br>
[그림 8-13] 절댓값 가지치기(MaP)와 이동 가지치기(MvP)에서 남은 가중치

- 트랜스포머스는 기본적으로 가지치기 방법을 제공하지 않음.
- 원래 이동 가지치기를 확장한 방법이 [`nn_pruning`(Neural Networks Block Movement Pruning)](https://huggingface.github.io/nn_pruning/) 라이브러리에 구현되어 있음.
- 성능 최적화에 관심이 있다면 허깅페이스 [🤗 Optimum](https://huggingface.co/docs/optimum/v1.2.1/en/index)을 시도해보는 것도 좋겠다.

## 결론
#### 작지만 빠르고 정확도 높은 모델이 필요하다면?
#### 예측 속도를 높이고 트랜스포머 모델의 메모리 사용량을 줄이는 기술 4 가지:
- 지식 정제(Knowledge Distillation)
- 양자화(Quantization)
- ONNX 포맷과 ONNX 런타임(ORT)을 사용한 그래프 최적화(Graph Optimization)
- 가지치기(Pruning)
  - 트랜스포머 모델의 저장 크기를 줄이는 가장 효과적인 방법이지만,
  - 하드웨어가 희소행렬 연산에 최적화되어야 가지치기의 효과를 제대로 볼 수 있다.

그러나, 레이턴시 요구사항을 맞추려고 해쓰고 있거나, 모델이 컴퓨팅 자원을 독식하고 있다면 이러한 기법 중 하나를 시도해보기를 권한다.///